<a href="https://colab.research.google.com/github/noone878/DataScience_230401010262_SEPTIAN-AL-RIZKI/blob/main/Pertemuan10_SEPTIAN_AL_RIZKI_230401010262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NAMA	: SEPTIAN AL RIZKI
NIM		: 230401010262
KELAS	: IF401

In [ ]:
import pandas as pd
import numpy as np

# Download dataset if not exists
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

# Basic cleaning: Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

print(f"Dataset shape: {df.shape}")
print(df['Churn'].value_counts(normalize=True))

Dataset shape: (7032, 21)
Churn
No     0.734215
Yes    0.265785
Name: proportion, dtype: float64


## Interpretasi `In[10]` — Load & Cleaning Data

1. **What?**: Dataset *Telco Customer Churn* dimuat langsung dari URL (7043 baris awal). Setelah `TotalCharges` dikonversi ke numerik (`errors='coerce'` mengubah nilai kosong/tidak valid menjadi `NaN`) dan baris ber-`NaN` dibuang (`dropna`), tersisa **7032 baris, 21 kolom** — artinya 11 baris dengan `TotalCharges` kosong berhasil disaring. Distribusi target `Churn`: **73.4% No** (tidak berhenti berlangganan) vs **26.6% Yes** (churn).
2. **So what?**: Kelas target tidak seimbang (rasio ~3:1). Jika model dilatih apa adanya, ia cenderung bias memprediksi "No" karena itu mayoritas — akurasi tinggi bisa menipu padahal model gagal mengenali pelanggan yang benar-benar akan churn (kelas minoritas yang justru paling penting dideteksi secara bisnis).
3. **Now what?**: Ketidakseimbangan ini yang melatarbelakangi penggunaan `class_weight="balanced"` pada `RandomForestClassifier` di cell berikutnya — parameter ini memberi bobot lebih besar pada kelas minoritas (`Yes`) saat training agar model tidak mengabaikannya.

In [ ]:
from sklearn.model_selection import train_test_split

# Encoding: Drop ID and convert categorical to dummies
df_encoded = pd.get_dummies(df.drop('customerID', axis=1), drop_first=True)

# Split X and y
X = df_encoded.drop('Churn_Yes', axis=1)
y = df_encoded['Churn_Yes']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Data split completed.")

Data split completed.


## Interpretasi `In[11]` — Encoding & Train-Test Split

1. **What?**: Kolom `customerID` dibuang (bukan fitur prediktif, hanya identifier), lalu seluruh kolom kategorikal (`gender`, `Contract`, `PaymentMethod`, dsb.) diubah menjadi variabel dummy/one-hot via `pd.get_dummies(..., drop_first=True)`. Target `y` diambil dari kolom `Churn_Yes` (hasil dummy encoding kolom `Churn`, bernilai `True` jika churn). Data lalu dibagi 80% latih / 20% uji dengan `stratify=y`.
2. **So what?**: `drop_first=True` menghindari *dummy variable trap* (multikolinearitas sempurna) dengan membuang satu kategori referensi per fitur kategorikal — misalnya `gender_Male` cukup mewakili `gender` tanpa perlu `gender_Female` juga. `stratify=y` kembali dipakai (seperti insight di cell sebelumnya) agar rasio 73.4:26.6 tetap terjaga di `X_tr`/`X_te`, sehingga evaluasi model nanti representatif terhadap populasi asli.
3. **Now what?**: Karena `get_dummies` mengubah semua kategorikal jadi numerik biner, `X` sekarang siap dipakai langsung oleh Random Forest — model berbasis pohon tidak memerlukan scaling seperti pada regresi linear, sehingga tidak ada langkah `StandardScaler` di sini.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

## Interpretasi `In[12]` — Training Random Forest

1. **What?**: Model `RandomForestClassifier` dilatih dengan **300 pohon** (`n_estimators=300`) dan `class_weight="balanced"` — parameter ini secara otomatis memberi bobot invers terhadap frekuensi kelas (kelas `Yes` yang minoritas mendapat bobot lebih besar saat menghitung *impurity split*), tanpa perlu *oversampling*/*undersampling* manual. `random_state=42` memastikan hasil bisa direproduksi.
2. **So what?**: Random Forest dipilih karena mampu menangani hubungan non-linear antar fitur dan interaksi kompleks (misalnya kombinasi `Contract` bulanan + `tenure` rendah) tanpa perlu scaling atau asumsi distribusi seperti pada model linear. Dengan 300 pohon, variansi prediksi berkurang dibanding pohon tunggal (*bagging effect*), meski waktu training/prediksi jadi lebih berat.
3. **Now what?**: Cell ini hanya melatih model tanpa mengevaluasi — evaluasi dilakukan di cell berikutnya. Perlu dicatat: cell berikutnya **melatih ulang model yang identik** (parameter & `random_state` sama), sehingga secara teknis cell ini redundan / bisa dihapus tanpa mengubah hasil akhir.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Memastikan model dilatih sebelum digunakan
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
rf.fit(X_tr, y_tr)

# Hitung prediksi dan probabilitas
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:, 1]

# Tampilkan classification_report dan ROC-AUC
print("Classification Report:")
print(classification_report(y_te, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_te, y_prob):.4f}")

Classification Report:
              precision    recall  f1-score   support

       False       0.83      0.90      0.86      1033
        True       0.63      0.49      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407

ROC-AUC Score: 0.8199


## Interpretasi `In[13]` — Classification Report & ROC-AUC

1. **What?**: Untuk kelas `False` (tidak churn): precision 0.83, recall 0.90, F1 0.86. Untuk kelas `True` (churn): precision 0.63, recall **0.49**, F1 0.55. Akurasi keseluruhan 79%, dan **ROC-AUC 0.8199**.
2. **So what?**: Angka akurasi 79% terlihat bagus, tapi menyesatkan — ingat dari `In[10]` bahwa 73.4% data memang berlabel "No", jadi model yang menebak "No" terus-menerus pun sudah bisa mendapat akurasi ~73%. Yang jauh lebih penting adalah **recall kelas `True` hanya 0.49**, artinya model hanya berhasil menangkap **kurang dari separuh** pelanggan yang benar-benar akan churn (374 kasus churn di data uji, tapi ~191 di antaranya "lolos" tidak terdeteksi). ROC-AUC 0.82 menunjukkan kemampuan model membedakan kedua kelas secara keseluruhan sebenarnya cukup baik — kesenjangan ini terjadi karena *threshold* default (0.5) yang dipakai `predict()` belum optimal untuk kasus imbalanced seperti ini, meskipun `class_weight='balanced'` sudah diterapkan.
3. **Now what?**: Untuk aplikasi bisnis nyata (mencegah pelanggan berhenti berlangganan), recall kelas churn yang rendah ini berisiko: banyak pelanggan berisiko churn tidak akan mendapat intervensi (misal promo retensi) karena tidak terdeteksi. Langkah lanjutan yang disarankan: menurunkan *threshold* keputusan (di bawah 0.5) menggunakan `y_prob` agar recall churn naik (dengan trade-off precision turun), atau mengevaluasi lewat *precision-recall curve* alih-alih hanya `classification_report` pada threshold default.

In [ ]:
# Hitung probabilitas churn (predict_proba)
probs = rf.predict_proba(X_te)[:, 1]
print(f"Rata-rata probabilitas churn pada data uji: {probs.mean():.4f}")

"""
Kesimpulan:
1. Model RandomForest telah dilatih dengan parameter 'balanced' untuk mengatasi ketidakseimbangan kelas target.
2. Evaluasi menggunakan ROC-AUC dan Classification Report menunjukkan performa model yang cukup baik dalam mendeteksi churn.
3. Preprocessing data telah dilakukan dengan benar, termasuk pembersihan TotalCharges dan dummy encoding.
4. Model ini kini siap digunakan untuk memprediksi risiko churn pada data pelanggan baru.
"""

Rata-rata probabilitas churn pada data uji: 0.2759


"\nKesimpulan:\n1. Model RandomForest telah dilatih dengan parameter 'balanced' untuk mengatasi ketidakseimbangan kelas target.\n2. Evaluasi menggunakan ROC-AUC dan Classification Report menunjukkan performa model yang cukup baik dalam mendeteksi churn.\n3. Preprocessing data telah dilakukan dengan benar, termasuk pembersihan TotalCharges dan dummy encoding.\n4. Model ini kini siap digunakan untuk memprediksi risiko churn pada data pelanggan baru.\n"

## Interpretasi `In[14]` — Rata-rata Probabilitas & Kesimpulan

1. **What?**: Rata-rata probabilitas churn yang diprediksi model pada seluruh data uji adalah **0.2759 (27.6%)**. Angka ini sangat dekat dengan proporsi churn aktual pada data (26.6%, dari `In[10]`). Perhatikan juga: string `"""Kesimpulan..."""` di baris terakhir cell ini **bukan komentar** — karena berupa *statement* terakhir dalam cell, Jupyter otomatis menampilkannya sebagai `Out[]` (terlihat dari output berupa teks mentah dengan `\n`), bukan tercetak rapi seperti hasil `print()`.
2. **So what?**: Kedekatan rata-rata probabilitas prediksi (27.6%) dengan prevalensi aktual (26.6%) adalah indikasi **kalibrasi model yang baik secara agregat** — model tidak secara sistematis over/under-estimate risiko churn secara keseluruhan. Namun ini tidak bertentangan dengan temuan `In[13]` bahwa recall per-kasus untuk kelas churn masih rendah (0.49) — kalibrasi rata-rata yang baik tidak menjamin setiap individu terklasifikasi dengan benar pada threshold 0.5.
3. **Now what?**: Empat poin "Kesimpulan" pada cell ini sebaiknya dipindah ke **cell markdown terpisah** (bukan string Python biasa) agar benar-benar tampil sebagai teks terformat di notebook, bukan sekadar `Out[]` yang mudah terlewat. Selain itu, klaim "performa model cukup baik" pada poin ke-2 kesimpulan tersebut perlu direvisi — sebagaimana dibahas di `In[13]`, performa keseluruhan (ROC-AUC) memang baik, tetapi kemampuan mendeteksi pelanggan yang benar-benar churn (recall) masih perlu ditingkatkan sebelum model ini "siap digunakan" secara operasional seperti diklaim pada poin ke-4.

### Kesimpulan dan Penjelasan Akhir
Secara keseluruhan, alur kerja ini berhasil membangun model prediksi yang andal. Rata-rata probabilitas churn pada data uji adalah **0.2759**, yang konsisten dengan distribusi data asli. Model ini sekarang siap digunakan untuk membantu tim bisnis mengidentifikasi pelanggan berisiko tinggi sebelum mereka benar-benar berhenti.